# Milestone 4 lab, notebook 1 of 7: Load first, transform later

## Where we are

| Notebook | What you do |
|---|---|
| **1. Load first, transform later** (you are here) | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

**Before you start,** open [`milestone_4_flow.html`](../milestone_4_flow.html) in a browser.

Milestone 3 ended with the right numbers and a problem you cannot see in a number. Every rule of the report lives in a Python string. Look at where the rule "a cancelled order is not a sale" was written.

In [1]:
from lab_helpers import MILESTONE_DIR, dbt, lab_warehouse, load_only, money, show_doc, table

old_code = (MILESTONE_DIR.parent / "m3_safe_to_run_twice" / "daily_orders" / "report.py").read_text()
start = old_code.index('INSERT_DAYS_SQL = """')
print(old_code[start : old_code.index('"""', start + 30) + 3])

INSERT_DAYS_SQL = """
INSERT INTO analytics.daily_orders
SELECT
    order_date,
    city,
    count(*) FILTER (WHERE status <> 'cancelled'),
    count(*) FILTER (WHERE status = 'cancelled'),
    coalesce(sum(total_cents - refunded_cents) FILTER (WHERE status <> 'cancelled'), 0),
    coalesce(sum(refunded_cents) FILTER (WHERE status <> 'cancelled'), 0)
FROM core.orders
WHERE order_date = ANY(%(days)s)
  AND NOT is_deleted                    -- a deleted order is not counted
GROUP BY order_date, city
"""


That is a business rule, inside a program. A finance analyst who knows SQL well cannot find it, cannot read the file it is in with confidence, and cannot change it.

This milestone moves every rule out of Python.

## Step 1: Python does less

From now on Python does two jobs: it **extracts** from the order system, and it **loads** what arrived. Run that for Monday and Tuesday.

In [2]:
con = lab_warehouse(fresh=True)
load_only(con, "2026-06-08", "2026-06-09")

table(con, """
    SELECT table_schema AS schema, table_name AS "table"
    FROM information_schema.tables
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema') ORDER BY 1, 2
""")

Landed 226 raw rows for the morning of 2026-06-08.
Landed 102 raw rows for the morning of 2026-06-09.


,schema,table
0,ops,load_runs
1,ops,report_changes
2,ops,watermarks
3,raw,order_deletes
4,raw,orders_landing


Five tables, and none of them is clean. There is no orders table and no report. Python landed the raw rows, saved its watermark, wrote its log, and stopped.

## Step 2: The warehouse does the rest

The transformations are in a folder called `dbt_project`. One command builds them all.

In [3]:
dbt("build")

1 of 38 OK created sql view model staging.stg_order_deletes .................... [CREATE VIEW]
2 of 38 OK loaded seed file core.cities ........................................ [INSERT 3]
3 of 38 PASS source_not_null_raw_orders_landing_landing_id ..................... [PASS]
4 of 38 PASS source_not_null_raw_orders_landing_payload ........................ [PASS]
5 of 38 PASS source_unique_raw_orders_landing_landing_id ....................... [PASS]
6 of 38 PASS not_null_stg_order_deletes_order_id ............................... [PASS]
7 of 38 PASS unique_stg_order_deletes_order_id ................................. [PASS]
8 of 38 PASS not_null_cities_city .............................................. [PASS]
9 of 38 PASS unique_cities_city ................................................ [PASS]
10 of 38 OK created sql view model staging.stg_orders .......................... [CREATE VIEW]
11 of 38 PASS not_null_stg_orders_order_id ..................................... [PASS]
12 of 38 PASS 

Read that output once, slowly. Each line is one thing dbt did, in order:

- `seed`: it loaded a small CSV file as a table.
- `view model` and `table model`: it built a table or a view from a `SELECT`.
- `test`: it ran a check on a table. `PASS` means the check found nothing wrong.

Now look at the warehouse again.

In [4]:
table(con, """
    SELECT table_schema AS schema, table_name AS "table", lower(table_type) AS kind
    FROM information_schema.tables
    WHERE table_schema IN ('staging', 'core', 'marts') ORDER BY 1, 2
""")

,schema,table,kind
0,core,cities,base table
1,core,order_items,base table
2,core,orders,base table
3,marts,daily_orders,base table
4,staging,stg_order_deletes,view
5,staging,stg_order_items,view
6,staging,stg_orders,view


In [5]:
table(con, """
    SELECT order_date, sum(orders) AS orders, sum(revenue_cents) AS revenue_cents
    FROM marts.daily_orders WHERE order_date IN ('2026-06-07', '2026-06-06', '2026-06-05')
    GROUP BY order_date ORDER BY order_date DESC
""")

,order_date,orders,revenue_cents
0,2026-06-07,30,80711
1,2026-06-06,33,78671
2,2026-06-05,37,97104


Sunday: 30 orders and $807.11. Friday: $971.04, with the refund. The same numbers as milestone 3. Not one line of Python worked them out.

## Step 3: A new word, ELT

Look at the order of the work.

| | Milestone 1 | Now |
|---|---|---|
| 1 | **E**xtract from the API | **E**xtract from the API |
| 2 | **T**ransform in Python (pandas) | **L**oad the raw rows into the warehouse |
| 3 | **L**oad the result | **T**ransform inside the warehouse (SQL) |
| The name | **ETL** | **ELT** |

> "ETL transforms the data and then loads only the result. ELT loads the raw data and then transforms it inside the warehouse."

### Compare them on the same data

You ran both now. Here is what each one left in the warehouse for the morning of 8 June. The left column is from milestone 1's lab.

In [6]:
show_doc("HLD.md", "2. What changed since milestone 3")

**From `docs/HLD.md`**

> ## 2. What changed since milestone 3
> 
> | Part | Milestone 3 | Milestone 4 |
> |---|---|---|
> | Where the transform is written | SQL inside Python strings | One `.sql` file per table (**model**) |
> | Who can change a rule | A programmer | Anyone who reads SQL |
> | The clean orders table | Kept up to date with `MERGE` | Rebuilt from raw on every run |
> | The order items | Dropped | A table of their own |
> | The report | Rebuilt for the days a run touched | An **incremental model**: rebuilds the newest days |
> | Checks on the data | None | **Table tests**, run after each table is built |
> | When a check fails | Does not apply | The run fails. The report keeps its last good numbers |
> | Old data | Nobody notices | A **freshness** check |

In [7]:
landed = table(con, """
    SELECT (SELECT count(*) FROM raw.orders_landing WHERE run_id = '2026-06-08_1') AS raw_orders,
           (SELECT count(*) FROM raw.orders_landing, jsonb_array_elements(payload->'items') WHERE run_id = '2026-06-08_1') AS raw_item_lines
""").iloc[0]

import pandas as pd
pd.DataFrame([
    {"": "Orders in the warehouse", "ETL (milestone 1)": 226, "ELT (now)": int(landed["raw_orders"])},
    {"": "Item lines in the warehouse", "ETL (milestone 1)": 0, "ELT (now)": int(landed["raw_item_lines"])},
    {"": "Sunday's report", "ETL (milestone 1)": "30 orders, $807.11", "ELT (now)": "30 orders, $807.11"},
    {"": "Where the rules are", "ETL (milestone 1)": "Python", "ELT (now)": "SQL files"},
    {"": "A new question about products", "ETL (milestone 1)": "Call the API again", "ELT (now)": "One SQL query"},
])

,,ETL (milestone 1),ELT (now)
0,Orders in the warehouse,226,226
1,Item lines in the warehouse,0,759
2,Sunday's report,"30 orders, $807.11","30 orders, $807.11"
3,Where the rules are,Python,SQL files
4,A new question about products,Call the API again,One SQL query


The report is the same. Everything around it is different. Ask the question milestone 1 could not answer.

In [8]:
table(con, """
    SELECT i.product_name, sum(i.quantity) AS units_sold
    FROM core.order_items AS i
    JOIN core.orders AS o USING (order_id)
    WHERE o.order_date = '2026-06-07' AND o.status <> 'cancelled' AND i.item_status = 'ok'
    GROUP BY 1 ORDER BY units_sold DESC, 1 LIMIT 5
""")

,product_name,units_sold
0,Basmati rice 1kg,12
1,Butter 250g,12
2,Chicken breast 500g,11
3,Baby spinach,10
4,Mozzarella,9


## Step 4: Change a rule, without Python

With ELT you can change your mind. Drop **everything** dbt built, as if a rule had been wrong for a month.

In [9]:
for schema in ["staging", "core", "marts"]:
    con.execute(f"DROP SCHEMA {schema} CASCADE")
print("Dropped every clean table and the report. The raw tables are untouched.")

dbt("build", show=False)
table(con, "SELECT count(*) AS orders FROM core.orders")

Dropped every clean table and the report. The raw tables are untouched.


,orders
0,251


It all came back, from the raw table alone. The order system was never asked.

With ETL, the raw answer was gone. A wrong rule meant reading the source again, and the source may have changed since.

ELT is not always better. It is the usual choice when the warehouse is cheap to fill and good at SQL. ETL still wins when data must be removed **before** it lands, such as a customer's name you are not allowed to store.

In [10]:
con.close()

## A new word

| Word | Meaning | Here |
|---|---|---|
| **ELT** | Extract, load, transform. It loads the raw data and then transforms it inside the warehouse | Python lands raw rows. dbt builds every table after that |

The definition is from the D8Loop chapter [ETL vs ELT](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/etl-vs-elt).

## What you now know

- Python now only extracts and loads. Every transformation is SQL in the warehouse.
- ELT gave the same report as ETL, and kept everything needed for the next question.
- The clean tables can be dropped and rebuilt from raw, with no call to the source.

**Say it in an interview:** "I started with ETL in pandas and moved to ELT: land the raw JSON, then transform with dbt in the warehouse. The report was identical. The difference showed when a new question came: with ETL I had dropped the item lines before loading, with ELT I wrote one more model."

Next: [notebook 2](02_your_first_model.ipynb) opens the `dbt_project` folder and writes a model.